In [1]:
# Importa las bibliotecas necesarias
import cv2
import numpy as np
import os
from matplotlib import pyplot as plt

In [11]:
# Directorio que contiene las imágenes
image_folder = '../docker_vm_test/02_preds/czi3channels_ac_open_detector_br0%_patches/images'

In [12]:
# Lee todas las imágenes en la carpeta
def load_images_from_folder(folder):
    images = []
    for filename in os.listdir(folder):
        img = cv2.imread(os.path.join(folder, filename))
        if img is not None:
            images.append(img)
    return images

In [31]:
# Función para coser dos imágenes
def stitch_images(img1, img2):
    # Convertir las imágenes a escala de grises
    gray1 = cv2.cvtColor(img1, cv2.COLOR_BGR2GRAY)
    gray2 = cv2.cvtColor(img2, cv2.COLOR_BGR2GRAY)
    
    # Detectar puntos clave y descriptores usando SIFT
    sift = cv2.SIFT_create()
    keypoints1, descriptors1 = sift.detectAndCompute(gray1, None)
    keypoints2, descriptors2 = sift.detectAndCompute(gray2, None)
    
    # Emparejar los descriptores usando FLANN
    FLANN_INDEX_KDTREE = 1
    index_params = dict(algorithm=FLANN_INDEX_KDTREE, trees=5)
    search_params = dict(checks=50)
    flann = cv2.FlannBasedMatcher(index_params, search_params)
    matches = flann.knnMatch(descriptors1, descriptors2, k=2)
    
    # Filtrar buenos emparejamientos
    good_matches = []
    for m, n in matches:
        if m.distance < 0.75 * n.distance:
            good_matches.append(m)
    
    # Encontrar la homografía
    if len(good_matches) > 10:
        src_pts = np.float32([keypoints1[m.queryIdx].pt for m in good_matches]).reshape(-1, 1, 2)
        dst_pts = np.float32([keypoints2[m.trainIdx].pt for m in good_matches]).reshape(-1, 1, 2)
        H, mask = cv2.findHomography(src_pts, dst_pts, cv2.RANSAC, 5.0)
    else:
        raise ValueError("No se encontraron suficientes emparejamientos")
    
    # Aplicar la homografía para alinear las imágenes
    height, width, channels = img2.shape
    result = cv2.warpPerspective(img1, H, (width + img1.shape[1], height))
    result[0:height, 0:width] = img2
    
    return result

In [32]:
# Cargar todas las imágenes de la carpeta
images = load_images_from_folder(image_folder)

In [33]:
# Cosido de todas las imágenes
if len(images) > 1:
    stitched_image = images[0]
    for i in range(1, len(images)):
        stitched_image = stitch_images(stitched_image, images[i])
else:
    stitched_image = images[0]

error: OpenCV(4.10.0) /io/opencv/modules/imgproc/src/imgwarp.cpp:3400: error: (-215:Assertion failed) (M0.type() == CV_32F || M0.type() == CV_64F) && M0.rows == 3 && M0.cols == 3 in function 'warpPerspective'


In [ ]:
# Mostrar la imagen cosida final
plt.figure(figsize=(20,10))
plt.imshow(cv2.cvtColor(stitched_image, cv2.COLOR_BGR2RGB))
plt.axis('off')
plt.show()

In [ ]:
cv2.imwrite('prueba.png' ,stitched_image)